In [ ]:
!pip install timesfm[torch]
!pip install mlflow scikit-learn pandas numpy

In [ ]:
!pip install -q dagshub mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 127MB/s]



In [ ]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=89610451-8d11-4a38-b149-4a7c9ee833f3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5a3937941b04a23842462078dd0e8d039ad3dc4e616a8de028462c6a94ec98d8




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [ ]:
import torch
import timesfm



torch.set_float32_matmul_precision("high")


timesfm_model = (
    timesfm
    .TimesFM_2p5_200M_torch
    .from_pretrained(
        "google/timesfm-2.5-200m-pytorch"
    )
)


print("TimesFM loaded successfully")

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

TimesFM loaded successfully


In [ ]:
timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=115,
        max_horizon=14,
        normalize_inputs=True
    )
)

In [ ]:
import numpy as np
import pandas as pd
import mlflow

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

import pandas as pd
import numpy as np

import torch
import timesfm

import mlflow
import mlflow.pytorch

from sklearn.metrics import mean_squared_error, mean_absolute_error



HORIZON = 14
CONTEXT = 115

mlflow.set_experiment("TimesFM_training")



train = pd.read_csv("train.csv")
features = pd.read_csv("features.csv")
stores = pd.read_csv("stores.csv")


train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])



df = (
    train
    .merge(stores, on="Store", how="left")
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
)


df["unique_id"] = (
    df["Store"].astype(str)
    + "_"
    + df["Dept"].astype(str)
)


df = df.sort_values(
    ["unique_id", "Date"]
)


print(df.shape)
print(df.head())



dates = df["Date"].sort_values().unique()

train_end = dates[int(len(dates)*0.8)]
val_end = dates[int(len(dates)*0.9)]


train_df = df[df["Date"] <= train_end]

val_df = df[
    (df["Date"] > train_end)
    &
    (df["Date"] <= val_end)
]


test_df = df[
    df["Date"] > val_end
]


print(
    train_df.shape,
    val_df.shape,
    test_df.shape
)


torch.set_float32_matmul_precision("high")


model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)


model.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT,
        max_horizon=HORIZON,
        normalize_inputs=True
    )
)

mlflow.set_experiment("TimesFM_training")

contexts = [
    52,
    78,
    104,
    115
]

horizons = [
    7,
    14
]



def calculate_metrics(y_true, y_pred):

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    wmae = (
        np.sum(
            np.abs(y_true - y_pred)
        )
        /
        np.sum(
            np.abs(y_true)
        )
    )

    return rmse, mae, wmae



def prepare_train_data(
    df,
    context,
    horizon
):

    inputs = []
    actuals = []


    for series in df["unique_id"].unique():

        values = (
            df[
                df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )



        if len(values) <= horizon:
            continue



        history = values[:-horizon]

        target = values[-horizon:]


        history = history[-context:]



        if len(history) < context:

            history = np.pad(
                history,
                (
                    context - len(history),
                    0
                ),
                mode="edge"
            )


        inputs.append(history)
        actuals.append(target)


    return (
        np.array(inputs),
        np.array(actuals)
    )




def prepare_validation_data(
    train_df,
    val_df,
    context,
    horizon
):

    inputs = []
    actuals = []


    for series in val_df["unique_id"].unique():


        history = (
            train_df[
                train_df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )


        future = (
            val_df[
                val_df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )


        if len(future) < horizon:
            continue


        history = history[-context:]


        if len(history) < context:

            history = np.pad(
                history,
                (
                    context - len(history),
                    0
                ),
                mode="edge"
            )


        inputs.append(history)

        actuals.append(
            future[:horizon]
        )


    return (
        np.array(inputs),
        np.array(actuals)
    )




results = []


for context in contexts:

    for horizon in horizons:


        print("\n" + "="*60)
        print(
            f"TIMESFM | CONTEXT={context} | HORIZON={horizon}"
        )
        print("="*60)


        timesfm_model.compile(
            timesfm.ForecastConfig(
                max_context=context,
                max_horizon=horizon,
                normalize_inputs=True
            )
        )




        train_inputs, train_actuals = prepare_train_data(
            train_df,
            context,
            horizon
        )


        print(
            "Train series:",
            len(train_inputs)
        )


        if len(train_inputs) == 0:
            print("No train series")
            continue



        train_predictions, _ = (
            timesfm_model.forecast(
                horizon=horizon,
                inputs=train_inputs
            )
        )


        train_predictions = np.array(
            train_predictions
        )


        train_rmse, train_mae, train_wmae = calculate_metrics(
            train_actuals.flatten(),
            train_predictions.flatten()
        )




        val_inputs, val_actuals = prepare_validation_data(
            train_df,
            val_df,
            context,
            horizon
        )


        print(
            "Validation series:",
            len(val_inputs)
        )


        if len(val_inputs) == 0:
            print("No validation series")
            continue



        val_predictions, _ = (
            timesfm_model.forecast(
                horizon=horizon,
                inputs=val_inputs
            )
        )


        val_predictions = np.array(
            val_predictions
        )


        val_rmse, val_mae, val_wmae = calculate_metrics(
            val_actuals.flatten(),
            val_predictions.flatten()
        )



        print("\nTRAIN")
        print("RMSE:", train_rmse)
        print("MAE:", train_mae)
        print("WMAE:", train_wmae)


        print("\nVALIDATION")
        print("RMSE:", val_rmse)
        print("MAE:", val_mae)
        print("WMAE:", val_wmae)




        with mlflow.start_run(
            run_name=f"TimesFM_context{context}_horizon{horizon}"
        ):


            mlflow.log_params(
                {
                    "model": "TimesFM-2.5-200M",
                    "type": "zero-shot",
                    "context": context,
                    "horizon": horizon,
                    "normalize_inputs": True
                }
            )


            mlflow.log_metrics(
                {
                    "train_RMSE": train_rmse,
                    "train_MAE": train_mae,
                    "train_WMAE": train_wmae,

                    "val_RMSE": val_rmse,
                    "val_MAE": val_mae,
                    "val_WMAE": val_wmae
                }
            )



        results.append(
            {
                "context": context,
                "horizon": horizon,

                "train_RMSE": train_rmse,
                "train_MAE": train_mae,
                "train_WMAE": train_wmae,

                "val_RMSE": val_rmse,
                "val_MAE": val_mae,
                "val_WMAE": val_wmae
            }
        )



results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "val_RMSE"
)


display(results_df)

(421570, 17)
       Store  Dept       Date  Weekly_Sales  IsHoliday Type    Size  \
87524     10     1 2010-02-05      40212.84      False    B  126512   
87525     10     1 2010-02-12      67699.32       True    B  126512   
87526     10     1 2010-02-19      49748.33      False    B  126512   
87527     10     1 2010-02-26      33601.22      False    B  126512   
87528     10     1 2010-03-05      36572.44      False    B  126512   

       Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  MarkDown4  \
87524        54.34       2.962        NaN        NaN        NaN        NaN   
87525        49.96       2.828        NaN        NaN        NaN        NaN   
87526        58.22       2.915        NaN        NaN        NaN        NaN   
87527        52.77       2.825        NaN        NaN        NaN        NaN   
87528        55.92       2.877        NaN        NaN        NaN        NaN   

       MarkDown5         CPI  Unemployment unique_id  
87524        NaN  126.442065        

,context,horizon,train_RMSE,train_MAE,train_WMAE,val_RMSE,val_MAE,val_WMAE
6,115,7,3665.403914,1460.416400,0.097565,2705.810181,1333.518497,0.084907
4,104,7,3665.035862,1455.005848,0.097203,2765.726997,1365.663596,0.086954
7,115,14,3808.832649,1649.873547,0.110096,2824.005493,1389.378972,0.082456
5,104,14,3600.695334,1565.945596,0.104495,2863.079837,1415.820885,0.084025
2,78,7,3814.720693,1521.323203,0.101633,3115.986144,1473.169217,0.093799
3,78,14,3651.097759,1582.760113,0.105617,3179.566469,1528.737948,0.090726
0,52,7,4385.519695,1735.529791,0.115944,3740.292962,1715.968073,0.109259
1,52,14,4395.758589,1893.386598,0.126346,3974.839012,1870.388216,0.111002


In [ ]:
import mlflow
import mlflow.pyfunc
import torch
import timesfm
import pickle
import os



BEST_CONTEXT = 115
BEST_HORIZON = 14




torch.set_float32_matmul_precision("high")


best_timesfm_model = (
    timesfm
    .TimesFM_2p5_200M_torch
    .from_pretrained(
        "google/timesfm-2.5-200m-pytorch"
    )
)


best_timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=BEST_CONTEXT,
        max_horizon=BEST_HORIZON,
        normalize_inputs=True
    )
)




config = {
    "model": "TimesFM-2.5-200M",
    "context": BEST_CONTEXT,
    "horizon": BEST_HORIZON,
    "normalize_inputs": True
}


with open(
    "timesfm_best_config.pkl",
    "wb"
) as f:

    pickle.dump(
        config,
        f
    )


print("Config saved")




with mlflow.start_run(
    run_name="TimesFM_Best_Context115_H14"
):

    mlflow.log_params(
        {
            "model": "TimesFM-2.5-200M",
            "type": "zero-shot",
            "context": BEST_CONTEXT,
            "horizon": BEST_HORIZON,
            "normalize_inputs": True
        }
    )


    mlflow.log_metrics(
        {
            "val_RMSE": 2824.005493,
            "val_MAE": 1389.378972,
            "val_WMAE": 0.082456
        }
    )


    mlflow.log_artifact(
        "timesfm_best_config.pkl"
    )


print("Best TimesFM pipeline saved")

Config saved
🏃 View run TimesFM_Best_Context115_H14 at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/10/runs/07e805e0ac8d49428b66460390b89aab
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/10
Best TimesFM pipeline saved


In [ ]:
import mlflow.pyfunc


class TimesFMWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):

        import timesfm

        self.model = (
            timesfm
            .TimesFM_2p5_200M_torch
            .from_pretrained(
                "google/timesfm-2.5-200m-pytorch"
            )
        )


        self.model.compile(
            timesfm.ForecastConfig(
                max_context=115,
                max_horizon=14,
                normalize_inputs=True
            )
        )


    def predict(
        self,
        context,
        model_input
    ):

        predictions, _ = self.model.forecast(
            horizon=14,
            inputs=model_input
        )

        return predictions

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [ ]:
import numpy as np
import pandas as pd
import torch
import timesfm
import mlflow
import mlflow.pyfunc

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)



CONTEXT = 115
HORIZON = 14


mlflow.set_experiment(
    "TimesFM_Final"
)




train = pd.read_csv("train.csv")
features = pd.read_csv("features.csv")
stores = pd.read_csv("stores.csv")


train["Date"] = pd.to_datetime(
    train["Date"]
)

features["Date"] = pd.to_datetime(
    features["Date"]
)




def preprocess_data(
    train,
    features,
    stores
):

    df = (
        train
        .merge(
            stores,
            on="Store",
            how="left"
        )
        .merge(
            features,
            on=[
                "Store",
                "Date",
                "IsHoliday"
            ],
            how="left"
        )
    )


    df["unique_id"] = (
        df["Store"].astype(str)
        +
        "_"
        +
        df["Dept"].astype(str)
    )


    return df.sort_values(
        [
            "unique_id",
            "Date"
        ]
    )



df = preprocess_data(
    train,
    features,
    stores
)




dates = (
    df["Date"]
    .sort_values()
    .unique()
)


train_end = dates[
    int(len(dates)*0.8)
]


val_end = dates[
    int(len(dates)*0.9)
]


train_df = df[
    df["Date"] <= train_end
]


val_df = df[
    (df["Date"] > train_end)
    &
    (df["Date"] <= val_end)
]


print(
    train_df.shape,
    val_df.shape
)



torch.set_float32_matmul_precision(
    "high"
)


timesfm_model = (
    timesfm
    .TimesFM_2p5_200M_torch
    .from_pretrained(
        "google/timesfm-2.5-200m-pytorch"
    )
)


timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT,
        max_horizon=HORIZON,
        normalize_inputs=True
    )
)




def create_validation_windows(
    train_df,
    val_df
):

    inputs = []
    actuals = []


    for series in val_df["unique_id"].unique():

        history = (
            train_df[
                train_df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )


        future = (
            val_df[
                val_df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )


        if len(future) < HORIZON:
            continue


        history = history[-CONTEXT:]


        if len(history) < CONTEXT:

            history = np.pad(
                history,
                (
                    CONTEXT-len(history),
                    0
                ),
                mode="edge"
            )


        inputs.append(history)

        actuals.append(
            future[:HORIZON]
        )


    return (
        np.array(inputs),
        np.array(actuals)
    )



val_inputs, val_actuals = create_validation_windows(
    train_df,
    val_df
)


print(
    "Validation input:",
    val_inputs.shape
)




val_predictions, _ = (
    timesfm_model.forecast(
        horizon=HORIZON,
        inputs=val_inputs
    )
)


val_predictions = np.array(
    val_predictions
)




y_true = val_actuals.flatten()
y_pred = val_predictions.flatten()


val_rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)


val_mae = mean_absolute_error(
    y_true,
    y_pred
)


val_wmae = (
    np.sum(
        np.abs(y_true-y_pred)
    )
    /
    np.sum(
        np.abs(y_true)
    )
)



print("Validation RMSE:", val_rmse)
print("Validation MAE:", val_mae)
print("Validation WMAE:", val_wmae)




with mlflow.start_run(
    run_name="TimesFM_Best_Context115_Horizon14"
):

    mlflow.log_params(
        {
            "model": "TimesFM-2.5-200M",
            "type": "zero-shot",
            "context": CONTEXT,
            "horizon": HORIZON,
            "normalize_inputs": True
        }
    )


    mlflow.log_metrics(
        {
            "val_RMSE": val_rmse,
            "val_MAE": val_mae,
            "val_WMAE": val_wmae
        }
    )


print("Validation run logged")

2026/07/12 17:25:16 INFO mlflow.tracking.fluent: Experiment with name 'TimesFM_Final' does not exist. Creating a new experiment.


(338738, 17) (41369, 17)
Validation input: (2803, 115)
Validation RMSE: 2824.0054933833076
Validation MAE: 1389.3789717835502
Validation WMAE: 0.08245573623100651
🏃 View run TimesFM_Best_Context115_Horizon14 at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/5c4e1959d50b4d6c9be8c637cd216a82
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
Validation run logged


In [ ]:
import mlflow.pyfunc
import pickle


class TimesFMPipeline(mlflow.pyfunc.PythonModel):

    def load_context(self, context):

        import timesfm
        import torch

        torch.set_float32_matmul_precision("high")


        self.context = 115
        self.horizon = 14


        self.model = (
            timesfm
            .TimesFM_2p5_200M_torch
            .from_pretrained(
                "google/timesfm-2.5-200m-pytorch"
            )
        )


        self.model.compile(
            timesfm.ForecastConfig(
                max_context=self.context,
                max_horizon=self.horizon,
                normalize_inputs=True
            )
        )



    def create_windows(
        self,
        df
    ):

        inputs = []
        ids = []


        for series in df["unique_id"].unique():

            history = (
                df[
                    df["unique_id"] == series
                ]
                ["Weekly_Sales"]
                .values
            )


            history = history[-self.context:]


            if len(history) < self.context:

                history = np.pad(
                    history,
                    (
                        self.context-len(history),
                        0
                    ),
                    mode="edge"
                )


            inputs.append(history)
            ids.append(series)


        return (
            np.array(inputs),
            ids
        )



    def predict(
        self,
        context,
        model_input
    ):

        df = model_input.copy()


        # create ID
        df["unique_id"] = (
            df["Store"].astype(str)
            +
            "_"
            +
            df["Dept"].astype(str)
        )


        df = df.sort_values(
            [
                "unique_id",
                "Date"
            ]
        )


        inputs, ids = self.create_windows(
            df
        )


        predictions, _ = (
            self.model.forecast(
                horizon=self.horizon,
                inputs=inputs
            )
        )


        result = pd.DataFrame(
            predictions
        )

        result["unique_id"] = ids


        return result

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [ ]:
with mlflow.start_run(
    run_name="TimesFM_Best_Pipeline_C115_H14"
):


    mlflow.log_params(
        {
            "model": "TimesFM-2.5-200M",
            "context": 115,
            "horizon": 14,
            "normalize_inputs": True,
            "type": "zero-shot"
        }
    )


    mlflow.pyfunc.log_model(
        artifact_path="timesfm_pipeline",
        python_model=TimesFMPipeline()
    )

2026/07/12 17:34:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 17:34:07 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/12 17:34:20 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


🏃 View run TimesFM_Best_Pipeline_C115_H14 at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/6aff6a13f9f44371ab444fc159a82629
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12


In [ ]:
val_lengths = val_df.groupby("unique_id").size()
import numpy as np
import pandas as pd

val_lengths = val_df.groupby("unique_id").size()

print(val_lengths.describe())

print(
    "Max validation length:",
    val_lengths.max()
)
from sklearn.metrics import mean_squared_error, mean_absolute_error



CONTEXT = 115
HORIZON = 14



def prepare_test_data(
    history_df,
    test_df,
    context,
    horizon
):

    inputs = []
    actuals = []


    for series in test_df["unique_id"].unique():

        history = (
            history_df[
                history_df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )


        future = (
            test_df[
                test_df["unique_id"] == series
            ]
            ["Weekly_Sales"]
            .values
        )



        if len(future) < horizon:
            continue



        history = history[-context:]



        if len(history) < context:

            history = np.pad(
                history,
                (
                    context - len(history),
                    0
                ),
                mode="edge"
            )


        inputs.append(history)

        actuals.append(
            future[:horizon]
        )


    return (
        np.array(inputs),
        np.array(actuals)
    )




history_df = pd.concat(
    [
        train_df,
        val_df
    ]
)


test_inputs, test_actuals = prepare_test_data(
    history_df,
    test_df,
    CONTEXT,
    HORIZON
)



print(
    "Test series:",
    len(test_inputs)
)

print(
    "Input shape:",
    test_inputs.shape
)




best_timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT,
        max_horizon=HORIZON,
        normalize_inputs=True
    )
)



test_predictions, _ = (
    best_timesfm_model.forecast(
        horizon=HORIZON,
        inputs=test_inputs
    )
)


test_predictions = np.array(
    test_predictions
)




y_true = test_actuals.flatten()

y_pred = test_predictions.flatten()



test_rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)


test_mae = mean_absolute_error(
    y_true,
    y_pred
)


test_wmae = (
    np.sum(
        np.abs(y_true - y_pred)
    )
    /
    np.sum(
        np.abs(y_true)
    )
)



print("==========================")
print("TIMESFM TEST RESULTS")
print("==========================")

print(
    "Test RMSE:",
    test_rmse
)

print(
    "Test MAE:",
    test_mae
)

print(
    "Test WMAE:",
    test_wmae
)
print("Max validation length:", val_lengths.max())

count    3123.000000
mean       13.246558
std         2.613849
min         1.000000
25%        14.000000
50%        14.000000
75%        14.000000
max        14.000000
dtype: float64
Max validation length: 14
Test series: 2803
Input shape: (2803, 115)
TIMESFM TEST RESULTS
Test RMSE: 2589.091516353661
Test MAE: 1257.2531915322427
Test WMAE: 0.07661769367522762
Max validation length: 14
